# 01 — Exploratory Data Analysis

Explores the **gold** star schema in `data/gold/warehouse.duckdb`.
Run the pipeline first: `python scripts/run_pipeline.py`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import duckdb, pandas as pd
from src.config import settings
con = duckdb.connect(str(settings.warehouse_path), read_only=True)
con.execute('SELECT table_name FROM information_schema.tables ORDER BY 1').df()

## Fact table shape & coverage

In [ ]:
fact = con.execute('SELECT * FROM hospital_activity_fact').df()
print(fact.shape)
print('date range:', fact.date_key.min(), '->', fact.date_key.max())
fact[['admissions','bed_occupancy_pct','waiting_list_size','median_wait_days','vacancy_rate','ae_attendances']].describe()

## National daily A&E demand & bed occupancy trend

In [ ]:
daily = (fact.groupby('date_key')
             .agg(ae=('ae_attendances','sum'), occ=('bed_occupancy_pct','mean'))
             .reset_index())
daily['date_key'] = pd.to_datetime(daily['date_key'])
ax = daily.set_index('date_key')[['ae']].plot(figsize=(10,3), title='National A&E attendances / day')
daily.set_index('date_key')[['occ']].plot(figsize=(10,3), title='Mean bed occupancy %')

## Waiting times by specialty

In [ ]:
q = '''SELECT s.specialty_name, AVG(f.median_wait_days) AS avg_wait, SUM(f.waiting_list_size) AS list
       FROM hospital_activity_fact f JOIN dim_specialty s USING (specialty_id)
       GROUP BY 1 ORDER BY avg_wait DESC'''
con.execute(q).df()

## Regional pressure & correlation of demand signals

In [ ]:
cols = ['bed_occupancy_pct','waiting_list_size','median_wait_days','vacancy_rate','ae_attendances','flu_index','covid_index','avg_temp_c']
fact[cols].apply(pd.to_numeric, errors='coerce').corr().round(2)

In [ ]:
con.close()